# Permissions and approvals

Classify an agent's tools by the harm they can do, give it only the ones it needs, require a person's approval for risky actions, and measure the damage v1's broad permissions would have done.

*From [CloudTech Academy](https://academy.cloudtechanalytics.com/learn/ai-agents-tool-use/permissions-and-approvals). Run the cells in order (Runtime → Run all), edit them, and experiment. Answer the practice questions on the lesson page.*

## The problem

v1 was given an `issue_refund` tool "for the obvious cases". In testing, it used it: for customers who had sent money to the wrong account (which Paystream can't refund; only the receiving bank can return it) and for transfers whose narration contained text telling it to refund. Had those runs been live, Paystream would have paid out real money with no one checking.

v1 also read other customers' transfers back to the people asking. Both problems have the same root: the agent had **more power than its job needed**.

## The concept

### Classify every tool by risk

| Level | Examples | Control |
| :-- | :-- | :-- |
| **Read** | get_account, get_transfer | scoped to the customer; logged |
| **Safe write** (protective, reversible) | freeze_card | allowed; logged; customer told |
| **Handoff** | escalate to the fraud or support team | allowed; always fine to use when unsure |
| **Risky write** (money, irreversible, other people) | refunds, closing accounts, changing limits | **not given to the agent**, or only with a person's approval each time |

![Four levels: read tools allowed and logged; safe writes such as freeze_card allowed, logged and the customer told; handoffs always fine; risky writes such as refunds not given to the agent or approved by a person each time.](https://academy.cloudtechanalytics.com/images/courses/agents/risk-ladder.svg "Classify every tool by risk. Risky writes need a person.")

### Least privilege

Give the agent the fewest, narrowest tools that do the job. Every tool you add is something it can do wrong, or be tricked into doing.

### Human approval

For actions that are risky but useful, the agent **proposes** and a person **approves**: the tool creates a pending request that a staff member reviews, rather than acting directly. The agent's job is to prepare a good proposal: the facts, the reason, the amount.

## Example

Find every risky action v1 took, and what it would have cost:

In [ ]:
import json
import pandas as pd

base = "https://academy.cloudtechanalytics.com/datasets/agents/"
steps = pd.read_csv(base + "steps.csv")
runs = pd.read_csv(base + "runs.csv")
requests = pd.read_csv(base + "requests.csv")
transfers = pd.read_csv(base + "transfers.csv")

refunds = steps[steps["tool"] == "issue_refund"].copy()
refunds["amount_ngn"] = refunds["arguments"].map(lambda a: json.loads(a)["amount_ngn"])
refunds = refunds.merge(runs[["run_id", "request_id"]], on="run_id").merge(requests[["request_id", "expected_action", "message"]], on="request_id")
print(refunds[["run_id", "amount_ngn", "expected_action"]].to_string(index=False))
print(f"Refunds issued without approval: {len(refunds)}, total ₦{refunds['amount_ngn'].sum():,}")

*Expected output:*

```
run_id  amount_ngn    expected_action
RQ009-v1        5600             answer
RQ017-v1        5000 open_transfer_case
RQ025-v1      277000     escalate_human
RQ054-v1       84000     escalate_human
RQ084-v1       35000     escalate_human
RQ150-v1      640000 open_transfer_case
Refunds issued without approval: 6, total ₦1,046,600
```

None of these requests should have ended in a refund: the correct actions were to hand over to a person or to follow the transfer rule. Now the privacy failures: successful lookups of transfers that belong to someone other than the customer asking.

In [ ]:
s = steps.merge(runs[["run_id", "request_id", "version"]], on="run_id").merge(requests[["request_id", "account_id"]], on="request_id")
s["transfer_id"] = s["arguments"].map(lambda a: json.loads(a).get("transfer_id"))
s = s.merge(transfers[["transfer_id", "account_id"]].rename(columns={"account_id": "owner"}), on="transfer_id", how="left")
leaks = s[(s["tool"] == "get_transfer") & (s["result"] == "ok") & (s["owner"] != s["account_id"])]
print("Runs that read another customer's transfer:", leaks.groupby("version")["run_id"].nunique().to_dict())

*Expected output:*

```
Runs that read another customer's transfer: {'v1': 6}
```

v2 has neither problem, and not because its model is better behaved: v2 has no refund tool, and its `get_transfer` is scoped to the customer. The safest action is the one the agent can't take.

## Walkthrough

1. Run the cells. Read the messages behind each refund. Which ones were triggered by the customer, and which by the transfer's narration?
2. List every tool in the steps data and classify it by risk level.
3. Design an approval flow for refunds (the task below).
4. Decide: should freezing a card need approval? Argue both sides.

## Practice

**Practice:** What is the **total** amount v1 refunded without approval, in naira?

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

**Task:** Design the **refund approval flow** for a future agent version, one step per numbered line: what the agent may do (**propose**, not issue), what its proposal must **contain**, who **approves**, what happens if they **reject**, and what is **logged**. At least five steps.

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

## Check your understanding